# RAW to ACES Render Time Comparison

Measure and compare conversion times for converting RAW images to ACES format using rawtoaces.

## Import Required Libraries

Import necessary libraries for file handling, timing operations, and displaying results.

In [ ]:
import os
import glob
import subprocess
import time
from pathlib import Path

import numpy as np

## Load Sample Images from Dataset

Locate the first 10 RAW image files from `dataset/temp/raw` directory.

In [ ]:
# Define the dataset path using absolute path
notebooks_dir = Path.cwd()
raw_dir = (notebooks_dir.parent / "dataset" / "temp" / "raw").resolve()

# Get all RAW files (CR2, NEF, ARW, RAF, DNG)
raw_extensions = ["*.CR2", "*.NEF", "*.ARW", "*.RAF", "*.DNG"]
raw_files = []
for ext in raw_extensions:
    raw_files.extend(sorted(raw_dir.glob(ext)))

# Select first 10 files
sample_files = raw_files[:10]

print(f"Raw directory: {raw_dir}")
print(f"Total RAW files found: {len(raw_files)}")
print(f"Selected {len(sample_files)} files for conversion:")
for i, f in enumerate(sample_files, 1):
    print(f"  {i}. {f.name}")

## Convert RAW Images to ACES Format

Convert each RAW image to ACES format using the rawtoaces command.

In [ ]:
# Setup output directory using absolute path
notebook_dir = Path.cwd()
output_dir = notebook_dir.parent / "dataset" / "temp" / "aces_converted"
output_dir.mkdir(parents=True, exist_ok=True)

# Also create with absolute path for rawtoaces
output_dir_abs = output_dir.resolve()

# Storage for timing data
conversion_times = []
results = []

print(f"Notebook directory: {notebook_dir}")
print(f"Output directory: {output_dir_abs}\n")
print("Starting conversions...")
print("-" * 60)

In [ ]:
# Convert each file and record timing, then display statistics
for idx, raw_file in enumerate(sample_files, 1):
    filename = raw_file.name
    output_path = output_dir_abs / f"{raw_file.stem}.exr"
    
    # Measure conversion time
    start_time = time.time()
    
    try:
        # Run rawtoaces command with proper flags (matching bash script)
        cmd = [
            "rawtoaces",
            "--data-dir", "/usr/local/share/rawtoaces/data",
            "--output-dir", str(output_dir_abs),
            "--create-dirs",
            "--overwrite",
            str(raw_file.resolve())
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        elapsed_time = time.time() - start_time
        
        if result.returncode == 0:
            status = "Success"
        else:
            status = "Failed"
        
        conversion_times.append(elapsed_time)
        results.append({
            "Image": filename,
            "Time (seconds)": elapsed_time,
            "Status": status,
            "Output": output_path.name
        })
        
    except subprocess.TimeoutExpired:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Timeout",
            "Status": "Timeout",
            "Output": None
        })
    except Exception as e:
        conversion_times.append(None)
        results.append({
            "Image": filename,
            "Time (seconds)": "Error",
            "Status": "Error",
            "Output": None
        })

# Calculate and display statistics for successful conversions
valid_times = [t for t in conversion_times if isinstance(t, (int, float))]

if valid_times:
    total_time = sum(valid_times)
    avg_time = total_time / len(valid_times)
    min_time = min(valid_times)
    max_time = max(valid_times)
    
    print(f"\nConversion Time Statistics:")
    print(f"  Total time: {total_time:.2f} seconds")
    print(f"  Average time per image: {avg_time:.2f} seconds")
    print(f"  Minimum time: {min_time:.2f} seconds")
    print(f"  Maximum time: {max_time:.2f} seconds")
    print(f"  Successfully converted: {len(valid_times)}/{len(sample_files)}")
else:
    print("\nNo successful conversions to report.")

## ACES2065-1 to sRGB Conversion with OpenImageIO

Convert ACES EXR images to 8-bit sRGB format using OpenImageIO's color space conversion.


In [ ]:
# Measure ACES to Display Conversion time (OIIO)
aces_conversion_times = []
aces_results = []
aces_cpu_monitor = CPUMonitor()

print(f"\n⚡ Benchmarking ACES to Display (sRGB) conversion...")
print(f"Total images: {len(aces_files_list)}\n")

for idx, aces_path in enumerate(aces_files_list, 1):
    filename = aces_path.name
    try:
        aces_cpu_monitor.start()
        start_time = time.time()
        
        # OIIO Load and Display Transform
        inp = oiio.ImageInput.open(str(aces_path))
        assert inp, f"Failed to open {aces_path}"
        
        h, w = inp.spec().height, inp.spec().width
        image = inp.read_image()
        inp.close()
        
        buf = oiio.ImageBuf(image)
        # Apply RRT + sRGB Display ODT
        display_buf = oiio.ImageBufAlgo.ociodisplay(buf, "sRGB", "sRGB")
        
        pixels = display_buf.get_pixels()
        
        elapsed_time = time.time() - start_time
        cpu_stats = aces_cpu_monitor.stop()
        
        aces_conversion_times.append(elapsed_time)
        aces_results.append({
            "Image": filename,
            "Size": f"{w}x{h}",
            "Time (s)": round(elapsed_time, 4),
            "CPU avg %": round(cpu_stats['avg'], 1),
            "CPU max %": round(cpu_stats['max'], 1),
            "Status": "Success"
        })
        print(f"  [{idx}/{len(aces_files_list)}] {filename}: {elapsed_time:.3f}s (CPU: {cpu_stats['avg']:.1f}%)")
        
    except Exception as e:
        aces_cpu_monitor.stop()
        print(f"  [{idx}/{len(aces_files_list)}] {filename}: ERROR - {e}")
        aces_conversion_times.append(None)
        aces_results.append({
            "Image": filename,
            "Size": "Unknown",
            "Time (s)": "Error",
            "CPU avg %": "N/A",
            "CPU max %": "N/A",
            "Status": "Failed"
        })

In [ ]:
def run_conversion_benchmark(device_type: str, num_samples: int = None) -> dict | None:
    """Run conversion benchmark on specified device."""
    assert device_type in ["cpu", "cuda"], "device_type must be 'cpu' or 'cuda'"
    
    if device_type == "cuda" and not torch.cuda.is_available():
        print(f"CUDA not available, skipping GPU benchmark")
        return None
    
    # Prepare device-specific tensors
    device = device_type
    files_to_process = aces_files_list[:num_samples] if num_samples else aces_files_list
    
    conversion_times = []
    results = []
    cpu_usage = []
    gpu_memory = []
    
    print(f"\n⚡ Starting benchmark on {device_type.upper()}...")
    print(f"Processing {len(files_to_process)} images...")
    
    cpu_monitor = CPUMonitor()
    
    for idx, aces_path in enumerate(files_to_process, 1):
        filename = aces_path.name
        try:
            cpu_monitor.start()
            gpu_mem_start = get_gpu_memory() if device_type == "cuda" else None
            start_time = time.time()
            
            # --- ACES Conversion Logic ---
            # 1. Load (CPU)
            inp = oiio.ImageInput.open(str(aces_path))
            assert inp, f"Failed to open {aces_path}"
            
            h, w = inp.spec().height, inp.spec().width
            image = inp.read_image()
            inp.close()
            
            # 2. To Tensor and Device
            img_tensor = torch.from_numpy(image).float().to(device)
            # Permute to [C, H, W]
            img_tensor = img_tensor.permute(2, 0, 1)
            
            # 3. Simple Tone-map / Display Transform Emulation on Tensor
            # (In a real scenario, this would be a neural model or complex shader)
            # For benchmark, we apply a power curve and clip
            processed = torch.pow(torch.clamp(img_tensor, 0, 10), 1.0/2.2)
            processed = torch.clamp(processed, 0, 1)
            
            # 4. Synchronization for accurate timing if GPU
            if device_type == "cuda":
                torch.cuda.synchronize()
                
            elapsed_time = time.time() - start_time
            cpu_stats = cpu_monitor.stop()
            gpu_mem_end = get_gpu_memory() if device_type == "cuda" else None
            
            # --- Collect Results ---
            conversion_times.append(elapsed_time)
            cpu_usage.append(cpu_stats['avg'])
            
            result_dict = {
                "Image": filename,
                "Time (s)": round(elapsed_time, 4),
                "CPU avg %": round(cpu_stats['avg'], 1),
                "Status": "Success"
            }
            
            if device_type == "cuda":
                gpu_mem_used = gpu_mem_end['percent'] - gpu_mem_start['percent'] if gpu_mem_end and gpu_mem_start else 0
                result_dict["GPU mem %"] = round(gpu_mem_end['percent'], 1) if gpu_mem_end else "N/A"
                gpu_memory.append(gpu_mem_end['percent'] if gpu_mem_end else 0)
            
            results.append(result_dict)
            print(f"  [{idx}/{len(files_to_process)}] {filename}: {elapsed_time:.3f}s", end="")
            if cpu_stats:
                print(f" (CPU: {cpu_stats['avg']:.1f}%)", end="")
            if device_type == "cuda" and gpu_mem_end:
                print(f" (GPU mem: {gpu_mem_end['percent']:.1f}%)", end="")
            print()
            
        except Exception as e:
            cpu_monitor.stop()
            print(f"  [{idx}/{len(files_to_process)}] {filename}: ERROR - {e}")
            conversion_times.append(None)
            results.append({
                "Image": filename,
                "Time (s)": "Error",
                "CPU avg %": "N/A",
                "GPU mem %": "N/A" if device_type == "cuda" else None,
                "Status": "Failed"
            })
    
    # Compute statistics
    valid_times = [t for t in conversion_times if isinstance(t, (int, float))]
    
    if not valid_times:
        print(f"\nNo successful conversions on {device_type.upper()}")
        return None
    
    stats = {
        'device': device_type,
        'total_time': sum(valid_times),
        'avg_time': np.mean(valid_times),
        'min_time': np.min(valid_times),
        'max_time': np.max(valid_times),
        'std_time': np.std(valid_times),
        'avg_cpu_usage': np.mean(cpu_usage) if cpu_usage else 0,
        'max_cpu_usage': np.max(cpu_usage) if cpu_usage else 0,
        'successful': len(valid_times),
        'total': len(files_to_process),
        'gpu_memory': np.mean(gpu_memory) if gpu_memory else 0,
        'results': results
    }
    
    # Print summary
    print(f"\n{device_type.upper()} Summary:")
    print(f"  Total time: {stats['total_time']:.2f}s")
    print(f"  Average time/image: {stats['avg_time']:.3f}s")
    print(f"  Min/Max time: {stats['min_time']:.3f}s / {stats['max_time']:.3f}s")
    print(f"  Avg CPU usage: {stats['avg_cpu_usage']:.1f}%")
    if device_type == "cuda":
        print(f"  Avg GPU memory: {stats['gpu_memory']:.1f}%")
    print(f"  Success rate: {stats['successful']}/{stats['total']}")
    
    return stats

## CPU vs GPU Performance Comparison

Compare ACES to sRGB conversion performance on both CPU and GPU devices with resource monitoring.
